# Are the "inert" cytokines really inert? Receptor expression in Parse

Several analyses use 13 cytokines assumed to have no effect on PBMCs as negative controls. A reviewer will ask whether their receptors are expressed (e.g. MET for HGF and FLT1/KDR/NRP1 for VEGF on monocytes, IGF1R on lymphocytes, NCAM1 as a GDNF co-receptor on NK cells).

This notebook reports receptor expression in the PBS pseudobulks of every cell type and defines a **strict inert set**: proteins whose signalling receptors are essentially absent (below `MAX_CPM` in every main cell type). The key analyses are then re-run with the strict set (next step).

In [1]:
import os
import numpy as np, pandas as pd
CACHE_DIR = "data/parse10m/cache"
RESULTS_DIR = "results/inert_receptors"
os.makedirs(RESULTS_DIR, exist_ok=True)
CONTROL = "PBS"
CELL_TYPES = ["B", "CD4 T", "CD8 T", "CD14 Mono", "CD16 Mono", "NK"]
MAX_CPM = 5          # a receptor counts as expressed if its mean CPM in PBS exceeds this in any main cell type

# signalling receptors / co-receptors of each assumed-inert protein
RECEPTORS = {
    "Noggin":   ["BMPR1A", "BMPR1B", "BMPR2", "ACVR1"],            # BMP antagonist: acts only via BMP signalling
    "Decorin":  ["EGFR", "MET", "IGF1R", "KDR"],                     # binds several receptor tyrosine kinases
    "PSPN":     ["GFRA4", "RET"],
    "GDNF":     ["GFRA1", "RET", "NCAM1"],
    "Megalin":  ["LRP2"],                                            # recombinant receptor fragment, no signalling receptor
    "PRL":      ["PRLR"],
    "EPO":      ["EPOR", "CSF2RB"],
    "TPO":      ["MPL"],
    "FGF-beta": ["FGFR1", "FGFR2", "FGFR3", "FGFR4"],
    "EGF":      ["EGFR", "ERBB2", "ERBB3", "ERBB4"],
    "HGF":      ["MET"],
    "VEGF":     ["FLT1", "KDR", "NRP1", "NRP2"],
    "IGF-1":    ["IGF1R", "INSR"],
}
sums = np.load(os.path.join(CACHE_DIR, "pb_sums.npy"))
meta = pd.read_pickle(os.path.join(CACHE_DIR, "pb_meta.pkl")).reset_index(drop=True)
genes = pd.Index(np.load(os.path.join(CACHE_DIR, "pb_genes.npy"), allow_pickle=True).astype(str))

## 1. Receptor expression (mean CPM over donors, PBS cells)

In [2]:
all_rec = sorted({g for v in RECEPTORS.values() for g in v})
missing = [g for g in all_rec if g not in genes]
print("receptor genes not in the data:", missing)
rows = {}
for ct in sorted(meta.cell_type.unique()):
    m = meta[(meta.cell_type == ct) & (meta.cytokine == CONTROL)]
    if m.empty: continue
    cpm = []
    for d, g in m.groupby("donor"):
        x = sums[g.index.to_numpy()].sum(0).astype(float); cpm.append(x / x.sum() * 1e6)
    cpm = np.mean(cpm, 0)
    rows[ct] = {g: cpm[genes.get_loc(g)] for g in all_rec if g in genes}
REC = pd.DataFrame(rows).T.round(1)
REC.to_csv(f"{RESULTS_DIR}/receptor_cpm_by_celltype.csv")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
REC

receptor genes not in the data: ['GFRA4']


,ACVR1,BMPR1A,BMPR1B,BMPR2,CSF2RB,EGFR,EPOR,ERBB2,ERBB3,ERBB4,FGFR1,FGFR2,FGFR3,FGFR4,FLT1,GFRA1,IGF1R,INSR,KDR,LRP2,MET,MPL,NCAM1,NRP1,NRP2,PRLR,RET
B,52.7,57.3,0.5,182.4,18.3,0.3,0.8,4.6,0.5,1.7,1.1,0.3,0.0,0.4,24.1,0.3,205.4,26.1,0.1,0.4,3.9,0.1,1.6,3.3,18.4,4.1,0.2
CD14 Mono,87.3,11.5,0.6,357.8,182.7,0.1,1.5,1.5,0.3,1.1,6.2,0.2,0.0,0.0,9.4,0.3,104.7,47.4,0.1,0.3,18.0,0.1,10.0,576.2,290.4,87.5,3.0
CD16 Mono,56.0,23.5,0.5,370.1,181.4,0.1,2.1,1.6,0.2,1.1,2.3,0.2,0.0,0.1,1.7,0.3,119.3,42.1,0.1,0.2,0.5,0.1,2.5,186.4,117.5,112.2,1.1
CD4 T,144.6,89.1,1.1,187.3,1.7,0.2,1.3,3.0,2.6,1.8,8.4,1.4,0.0,0.5,7.8,0.4,820.6,9.6,0.1,0.4,0.4,0.2,1.8,3.5,3.1,1.0,0.2
CD8 T,86.6,86.4,0.9,232.5,1.6,0.2,1.0,3.8,6.1,4.2,9.3,1.9,0.0,0.3,8.0,0.4,521.4,5.0,0.1,0.4,0.5,0.1,15.1,3.4,2.4,1.1,0.2
HSPC,128.2,10.1,20.4,144.7,197.7,0.1,13.4,4.1,0.4,2.8,2.5,11.2,0.1,0.2,2.0,0.4,366.3,102.7,3.9,2.0,1.1,40.7,8.5,18.1,13.7,5.1,1.3
MAIT,133.3,109.4,0.8,193.9,1.7,0.3,1.2,2.7,5.0,3.0,14.7,0.6,0.0,0.3,2.0,0.4,228.8,8.4,0.1,0.4,0.6,0.2,12.3,3.0,2.6,1.0,0.4
NK,22.6,79.8,0.8,265.2,1.5,0.2,0.9,11.2,0.9,2.8,7.6,0.4,0.0,0.3,4.4,0.3,428.9,2.7,0.1,0.5,0.4,0.3,782.4,3.3,2.1,0.9,0.2
NK CD56bright,32.8,38.6,0.6,196.8,3.2,0.2,1.0,4.2,2.1,4.2,3.4,0.3,0.0,0.5,2.4,0.2,302.0,11.3,0.3,0.5,1.0,0.2,2179.8,8.5,2.7,1.5,0.2
NKT,43.3,130.9,0.8,255.7,2.4,0.1,1.3,7.1,1.6,7.5,11.2,5.0,0.0,0.2,1.8,0.4,160.7,4.8,0.1,0.5,0.4,0.2,105.9,2.6,2.3,1.2,0.6


## 2. Per cytokine: highest receptor expression in the main cell types, and the strict set

In [3]:
rows = []
for cyto, recs in RECEPTORS.items():
    present = [g for g in recs if g in REC.columns]
    sub = REC.loc[[c for c in CELL_TYPES if c in REC.index], present]
    top = sub.max().sort_values(ascending=False)
    worst = top.index[0] if len(top) else None
    rows.append(dict(cytokine=cyto, receptors=", ".join(recs),
                     max_receptor=worst, max_cpm=float(top.iloc[0]) if len(top) else np.nan,
                     cell_type_of_max=sub[worst].idxmax() if worst else None,
                     receptors_expressed=", ".join(f"{g} ({sub[g].idxmax()} {sub[g].max():.0f})" for g in present if sub[g].max() > MAX_CPM)))
INERT_TABLE = pd.DataFrame(rows).set_index("cytokine")
# receptor genes absent from the data are treated as not expressed (flagged)
INERT_TABLE["note"] = np.where(INERT_TABLE.max_cpm.isna(), "no receptor gene in the data", "")
INERT_TABLE["max_cpm"] = INERT_TABLE.max_cpm.fillna(0.0)
INERT_TABLE = INERT_TABLE.sort_values("max_cpm")
INERT_TABLE["strict_inert"] = INERT_TABLE.max_cpm <= MAX_CPM
INERT_TABLE.to_csv(f"{RESULTS_DIR}/inert_receptor_summary.csv")
display(INERT_TABLE)
STRICT_INERT = INERT_TABLE.index[INERT_TABLE.strict_inert].tolist()
print("STRICT_INERT =", STRICT_INERT)

,receptors,max_receptor,max_cpm,cell_type_of_max,receptors_expressed,note,strict_inert
cytokine,,,,,,,
TPO,MPL,MPL,0.3,NK,,,True
Megalin,LRP2,LRP2,0.5,NK,,,True
PSPN,"GFRA4, RET",RET,3.0,CD14 Mono,,,True
FGF-beta,"FGFR1, FGFR2, FGFR3, FGFR4",FGFR1,9.3,CD8 T,FGFR1 (CD8 T 9),,False
EGF,"EGFR, ERBB2, ERBB3, ERBB4",ERBB2,11.2,NK,"ERBB2 (NK 11), ERBB3 (CD8 T 6)",,False
HGF,MET,MET,18.0,CD14 Mono,MET (CD14 Mono 18),,False
PRL,PRLR,PRLR,112.2,CD16 Mono,PRLR (CD16 Mono 112),,False
EPO,"EPOR, CSF2RB",CSF2RB,182.7,CD14 Mono,CSF2RB (CD14 Mono 183),,False
Noggin,"BMPR1A, BMPR1B, BMPR2, ACVR1",BMPR2,370.1,CD16 Mono,"BMPR1A (CD4 T 89), BMPR2 (CD16 Mono 370), ACVR...",,False


STRICT_INERT = ['TPO', 'Megalin', 'PSPN']
